In [1]:
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_extract, col, when, length 
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [2]:
spark = (
    SparkSession.builder.appName("Cleaning")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9100")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .getOrCreate()
)

In [3]:
minio_client = Minio(
    "localhost:9100",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False,
)

bucket_name = "pulse-bucket-1"

In [4]:
objects = minio_client.list_objects(bucket_name, prefix="mapped_", recursive=True)
dataframes = {}
for obj in objects:
    df = spark.read.csv(
        f"s3a://{bucket_name}/{obj.object_name}", header=True, inferSchema=True
    )
    object_name = obj.object_name.replace("mapped_", "").replace(".csv", "")
    dataframes[object_name] = df

In [5]:
for table in dataframes.keys():
    df = dataframes[table]

    for column in df.columns:
        if column.endswith("_id"):
            df = df.withColumn(
                column,
                when(
                    regexp_extract(col(column), r"(\d+)", 1) == "",
                    None,
                ).otherwise(regexp_extract(col(column), r"(\d+)", 1)),
            )
            df = df.withColumn(column, col(column).cast("int"))

    dataframes[table] = df
for table, dataframe in dataframes.items():
    print(f"Table: {table}")
    dataframe.show(3)

Table: addresses
+----------+------------------+--------------+-----------+--------------+
|address_id|              city|state_province|postal_code|       country|
+----------+------------------+--------------+-----------+--------------+
|      5555|North Derrickmouth|      Sevilla*|       2944|      Slovénie|
|      5938|             Husum|         Idaho|    PH0 6RX|        Rwanda|
|      5060|              NULL|          NULL|    00000  |United Kingdom|
+----------+------------------+--------------+-----------+--------------+
only showing top 3 rows

Table: categories
+-----------+----------+---------------+
|category_id|  category|   sub_category|
+-----------+----------+---------------+
|        695|  Colthing|      Wholesale|
|        659|Mirrorless|Limited Edition|
|        571|     Suits|      Wholesale|
+-----------+----------+---------------+
only showing top 3 rows

Table: customer_sessions
+----------+-----------+--------------------+--------------------+-----------+-------

In [6]:
dataframes["customers"].columns

['customer_id',
 'gender',
 'date_of_birth',
 'account_status',
 'address_id',
 'city',
 'state_province',
 'postal_code',
 'country',
 'account_created_at',
 'last_login_date',
 'is_active']

Merging  addresses table with Customers
and categories table with products if exists

In [ ]:
def merge():
      if not "addresses" in dataframes:
            print("Addresses DataFrame is missing.")

      if "addresses" in dataframes and "customers" in dataframes:
            dataframes["addresses"].createOrReplaceTempView("addresses")
            dataframes["customers"].createOrReplaceTempView("customers")

            customers = spark.sql("""
            SELECT c.customer_id, c.gender, c.date_of_birth, c.account_status,
                  a.city, a.state_province, a.postal_code, a.country,
                  c.account_created_at
            FROM customers c
            LEFT JOIN addresses a
                  ON c.address_id = a.address_id
            """)
            dataframes["customers"] = customers
            print("Merged addresses into customers.")
            dataframes.pop("addresses", None)

      if not "categories" in dataframes:
            print("Categories DataFrame is missing.")
            return
      
      if "categories" in dataframes and "products" in dataframes:
            dataframes["categories"].createOrReplaceTempView("categories")
            dataframes["products"].createOrReplaceTempView("products")

            products = spark.sql("""
            SELECT p.product_id, p.product_name, p.sku, cat.category, cat.sub_category,
                  p.brand, p.supplier_id, p.cost_price, p.sell_price, p.launch_date,
                  p.weight, p.dimensions, p.color, p.size, p.material
            FROM products p
            LEFT JOIN categories cat
                  ON p.category_id = cat.category_id
            """)
            dataframes["products"] = products
            print("Merged categories into products.")
            dataframes.pop("categories", None)
merge()

Addresses DataFrame is missing.
Categories DataFrame is missing.


# Cleaning null values and Duplicate values 

In [8]:
import pyspark.sql.functions as F

In [9]:
def check_dups():
    for name, df in dataframes.items():
        dup_rows = df.groupBy(*df.columns).count().filter("count > 1")
        row_count = dup_rows.count()
        print(f"The number of duplicate rows in {name} is: {row_count}")
check_dups()

The number of duplicate rows in customer_sessions is: 39
The number of duplicate rows in customers is: 26
The number of duplicate rows in inventory is: 20
The number of duplicate rows in marketing_campaigns is: 7
The number of duplicate rows in order_items is: 89
The number of duplicate rows in orders is: 19
The number of duplicate rows in payments is: 37
The number of duplicate rows in products is: 13
The number of duplicate rows in reviews is: 37
The number of duplicate rows in shopping_cart is: 43
The number of duplicate rows in suppliers is: 8
The number of duplicate rows in wishlist is: 36


In [10]:
def drop_dups():
    for table in dataframes.keys():
        dataframes[table] = dataframes[table].dropDuplicates()
drop_dups()


In [11]:
check_dups()

The number of duplicate rows in customer_sessions is: 0
The number of duplicate rows in customers is: 0
The number of duplicate rows in inventory is: 0
The number of duplicate rows in marketing_campaigns is: 0
The number of duplicate rows in order_items is: 0
The number of duplicate rows in orders is: 0
The number of duplicate rows in payments is: 0
The number of duplicate rows in products is: 0
The number of duplicate rows in reviews is: 0
The number of duplicate rows in shopping_cart is: 0
The number of duplicate rows in suppliers is: 0
The number of duplicate rows in wishlist is: 0


In [12]:
def drop_null_rows(table, col_name):
    if table in dataframes:
        df = dataframes[table]
        if col_name in df.columns:
            before = df.count()
            cleaned = df.filter(F.col(col_name).isNotNull())
            dataframes[table] = cleaned
            after = cleaned.count()
            print(f"Removed {before - after} rows from '{table}' where '{col_name}' is NULL")
        else:
            print(f"Column '{col_name}' not found in '{table}'")    
    else:
        print(f"Table '{table}' not found in dataframes")

Dropping all rows from all tables where primary key is null 

In [13]:
def drop_all_null_pk_rows():
    for table in dataframes.keys():
        drop_null_rows(table, dataframes[table].columns[0]) #in our schema 1st column of each table is its primary key
drop_all_null_pk_rows()

Removed 83 rows from 'customer_sessions' where 'session_id' is NULL
Removed 13 rows from 'customers' where 'customer_id' is NULL
Removed 26 rows from 'inventory' where 'inventory_id' is NULL
Removed 25 rows from 'marketing_campaigns' where 'campaign_id' is NULL
Removed 57 rows from 'order_items' where 'order_item_id' is NULL
Removed 48 rows from 'orders' where 'order_id' is NULL
Removed 58 rows from 'payments' where 'payment_id' is NULL
Removed 18 rows from 'products' where 'product_id' is NULL
Removed 34 rows from 'reviews' where 'review_id' is NULL
Removed 46 rows from 'shopping_cart' where 'cart_id' is NULL
Removed 13 rows from 'suppliers' where 'supplier_id' is NULL
Removed 27 rows from 'wishlist' where 'wishlist_id' is NULL


In [14]:
def check_nulls():
    for df in dataframes.values():
        null_counts = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns])
        null_counts.show()
check_nulls()

+----------+-----------+-------------+-----------+-----------+---------------+------------+---------------+---------------+---------------------+
|session_id|customer_id|session_start|session_end|device_type|referrer_source|pages_viewed|products_viewed|conversion_flag|cart_abandonment_flag|
+----------+-----------+-------------+-----------+-----------+---------------+------------+---------------+---------------+---------------------+
|         0|       1863|          154|        185|        185|            201|         178|            183|            151|                  100|
+----------+-----------+-------------+-----------+-----------+---------------+------------+---------------+---------------+---------------------+

+-----------+------+-------------+--------------+----+--------------+-----------+-------+------------------+
|customer_id|gender|date_of_birth|account_status|city|state_province|postal_code|country|account_created_at|
+-----------+------+-------------+--------------+--